# Capítulo 3: Procesos de Decisión de Markov (MDP) y Ecuaciones de Bellman

En este cuaderno resolvemos el MDP de OptiBot mediante el algoritmo de **Iteración de Valor (Value Iteration)** utilizando el operador de optimalidad de Bellman:

$$V_{k+1}(s) = \max_{a \in \mathcal{A}(s)} \sum_{s', r} p(s', r \mid s, a) \big[ r + \gamma V_k(s') \big]$$


In [ ]:
import numpy as np

# 1. Definición del MDP de OptiBot
estados = ["Muelle", "Pasillo", "Entrega", "Fin"]
num_estados = 3  # Estados activos (Fin es terminal con V=0)
gamma = 0.90
theta = 0.01

# Tabla de transiciones: [origen, accion, destino, recompensa, probabilidad]
transiciones = [
    # Muelle (S1)
    (0, "Carga Estandar", 1,  1.00, 1.00),
    (0, "Linea Rapida",   2,  0.10, 1.00),
    # Pasillo (S2)
    (1, "Modo Eco",       1,  0.30, 0.20),
    (1, "Modo Eco",       2,  1.00, 0.80),
    (1, "Modo Normal",    1,  0.50, 0.70),
    (1, "Modo Normal",    2,  0.30, 0.30),
    # Entrega (S3)
    (2, "Descarga Manual",3,  1.00, 0.85),
    (2, "Descarga Manual",1, -1.00, 0.15),
    (2, "Descarga Auto",  3,  1.00, 0.90),
    (2, "Descarga Auto",  1, -1.00, 0.10),
]

# 2. Inicialización de funciones de valor: V(s) = 0
V = np.zeros(4) # V[3] corresponde al estado terminal 'Fin' = 0

# 3. Bucle de Evaluación Iterativa / Value Iteration
iteracion = 0
while True:
    delta = 0.0
    iteracion += 1
    V_nuevo = V.copy()
    
    # Evaluar cada estado activo
    for s in range(num_estados):
        v_anterior = V[s]
        # Identificar acciones disponibles en el estado s
        acciones_s = sorted(list(set(a for s_orig, a, _, _, _ in transiciones if s_orig == s)))
        q_valores = []
        
        for accion in acciones_s:
            # Calcular suma de p * [r + gamma * V(s')]
            suma_q = sum(p * (r + gamma * V[dest]) 
                         for s_orig, a, dest, r, p in transiciones 
                         if s_orig == s and a == accion)
            q_valores.append(suma_q)
            
        V_nuevo[s] = max(q_valores)
        delta = max(delta, abs(v_anterior - V_nuevo[s]))
        
    V = V_nuevo
    if delta < theta:
        break

# 4. Extracción de la Política Óptima pi*(s)
politica_optima = {}
for s in range(num_estados):
    acciones_s = sorted(list(set(a for s_orig, a, _, _, _ in transiciones if s_orig == s)))
    mejor_q = -np.inf
    mejor_accion = None
    for accion in acciones_s:
        suma_q = sum(p * (r + gamma * V[dest]) 
                     for s_orig, a, dest, r, p in transiciones 
                     if s_orig == s and a == accion)
        if suma_q > mejor_q:
            mejor_q = suma_q
            mejor_accion = accion
    politica_optima[estados[s]] = (mejor_accion, mejor_q)

# 5. Visualización de resultados
print(f"=== CONVERGENCIA ALCANZADA EN {iteracion} ITERACIONES (theta = {theta}) ===")
for estado, (accion, valor) in politica_optima.items():
    print(f"  Estado {estado:8s} -> Acción Óptima: {accion:18s} | V*(s) = {valor:.4f}")
